# PPO — пилоты и основное обучение

Для пилотов в первой ячейке оставляю `MODE = "pilot"` и запускаю все ячейки. На сохранённых ответах сида 0 подготавливаю линейный критик и критик с MLP, затем запускаю два пилота по очереди. Сравнение критиков на отложенных темах храню в `ppo_exp028_v2.ipynb`.

Для основного обучения выбираю `MODE = "train"`, выставляю одну архитектуру критика для обоих запусков и указываю свой сид. Спецификация обновления политики — `docs/ppo.md`.

## 1. Сервер vLLM (терминал, карта 1) — как у GRPO

```bash
source ~/vllm085/bin/activate
cd ~/Diplom
VLLM_NO_USAGE_STATS=1 VLLM_USE_V1=0 VLLM_ALLOW_RUNTIME_LORA_UPDATING=True CUDA_VISIBLE_DEVICES=1 \
nohup python -m vllm.entrypoints.openai.api_server --model t-tech/T-lite-it-2.1 --revision d125c970c553de58fcee3c937d5e4867d4a448d8 \
    --served-model-name base --dtype float16 --enable-lora --max-lora-rank 32 --max-loras 1 \
    --max-model-len 25600 --gpu-memory-utilization 0.75 --generation-config vllm --port 8000 --seed 0 \
    > ~/vllm_ppo.log 2>&1 &
grep "startup complete" ~/vllm_ppo.log
```

## Настройки запуска

Проверяю доступность vLLM. В пилоте беру сид 0. Для сравнения по умолчанию делаю по пять шагов. При ограниченном времени могу поставить три. Проверка на одних validation-заданиях проходит на последнем шаге каждого пилота.

Для основного обучения выбираю в `CRITIC` линейный вариант (`"linear"`) или MLP (`"mlp"`). Если продолжаю основной запуск, задаю `RUN_DIR`.

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("WIKI_PROXY", "")
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from presentation_rl.training.common import model_commit
from presentation_rl.training.ppo import (
    PPOConfig, PREFIX_CRITICS, prepare_prefix_critics, run_prefix_pilots, train, warmup_generate,
)
from presentation_rl.training.vllm_client import VLLMClient

MODE = "pilot"  # "pilot" или "train"
PILOT_STEPS = 5
SEED = 0  # для основного запуска 0 или 1
CRITIC = None  # после пилота: "linear" или "mlp"
RUN_DIR = None  # для продолжения: путь к папке запуска
GENERATE_WARMUP = False  # включаю, когда ещё нет сохранённых эпизодов разогрева

if MODE not in ("pilot", "train"):
    raise ValueError("выбираю MODE: pilot или train")
if MODE == "pilot" and SEED != 0:
    raise ValueError("для пилотов беру SEED = 0")
if MODE == "train" and CRITIC not in PREFIX_CRITICS:
    raise ValueError("после пилота выбираю CRITIC: linear или mlp")
config = PPOConfig.load(ROOT / "configs" / "algorithms" / "ppo.yaml", seed=SEED)
WARMUP = ROOT / "runs" / "ppo" / f"warmup_seed{SEED}"
client = VLLMClient(config.vllm_url)
print(config.policy_model, "коммит:", model_commit(config.policy_model, config.policy_revision), "| сид", SEED)
print("vLLM", client.version(), client.models())

## 2. Проверки перед запуском

Запускаю тесты PPO, префиксных признаков и общих частей обучения. Память и время шага измеряю в пилоте на реальных ответах.

In [ ]:
tests = ["tests/test_ppo.py", "tests/test_ppo_torch.py", "tests/test_ppo_prefix.py",
         "tests/test_rl_torch.py", "tests/test_rl_common.py"]
subprocess.run([sys.executable, "-m", "pytest", "-q", *tests], cwd=ROOT, check=True)

## 3. Эпизоды разогрева

Использую готовые 640 ответов из `episodes.jsonl`. Если собираю разогрев с нуля, включаю `GENERATE_WARMUP`. Генерация продолжается с последнего готового пакета.

In [ ]:
if GENERATE_WARMUP:
    print(warmup_generate(config, ROOT, client))
episodes_file = WARMUP / "episodes.jsonl"
if not episodes_file.is_file():
    raise FileNotFoundError(f"Включаю GENERATE_WARMUP, чтобы получить {episodes_file}")
print(f"Эпизоды разогрева: {episodes_file}")

## 4. Разогрев критиков и два пилота

На всех 640 ответах сида 0 подготавливаю оба критика на одних префиксных признаках. Линейный обучаю четыре эпохи, MLP с 64 скрытыми нейронами — две. Сохраняю их отдельно и использую одинаковую исходную политику, очередь заданий и начальный снимок кеша в обоих пилотах.

Один шаг — 8 заданий по 8 ответов. На GPU 0 считаю опорные вероятности, одновременно на GPU 1 и CPU считаю награду и префиксные признаки. После каждого шага сохраняю состояние. При перезапуске продолжаю незаконченный пилот и читаю журналы готового.

Сравниваю ошибку критика до обновления, ошибку на первом действии, награду validation, время и долю отсечения. Оба журнала и итоговая сводка лежат в `runs/ppo_pilot/`.

In [ ]:
if MODE == "pilot":
    started = time.monotonic()
    pilots = run_prefix_pilots(config, ROOT, client, steps=PILOT_STEPS)
    for kind, row in pilots.items():
        critic, val = row["critic_before"], row["validation"]
        print(f"\n{PREFIX_CRITICS[kind]['label']}")
        print(f"MSE: {critic['all_mse']:.5f} | константа {critic['all_mse_const']:.5f} | "
              f"среднее по режиму {critic['all_mse_mode']:.5f}")
        print(f"Первое действие: MSE {critic['first_mse']:.5f} | среднее по режиму {critic['first_mse_mode']:.5f}")
        print(f"Validation: с поиском {val['search_reward']:.4f} | без поиска {val['plain_reward']:.4f}")
        print(f"Средний шаг {row['mean_step_seconds'] / 60:.2f} мин | "
              f"префиксные признаки {row['mean_prefix_seconds']:.1f} с | пик {row['peak_memory_gb']:.2f} ГиБ")
        print(f"Отсечено {row['mean_clip_active']:.3f} | максимум KL(old‖new) {row['max_kl_old_new']} | "
              f"пропущено обновлений {row['skipped_updates']}")
        print(row["run_dir"])
    print(f"Время текущего прогона {(time.monotonic() - started) / 60:.1f} мин")
    print(ROOT / "runs" / "ppo_pilot" / f"prefix_{PILOT_STEPS}_steps" / "comparison.json")

## 5. Основное обучение

После решения по пилоту выбираю одну архитектуру и применяю её к двум сидам. Для сида 0 читаю подготовленный критик из пилота. Для сида 1 обучаю выбранный критик на его собственном разогреве. Для продолжения беру состояние из `RUN_DIR`.

In [ ]:
if MODE == "train":
    trained_config = prepare_prefix_critics(config, ROOT, (CRITIC,))[CRITIC]
    print(train(trained_config, ROOT, run_dir=RUN_DIR, client=client))